# G2 — the SFT runs, retrained toward the gold label

The first pair of runs kept only trajectories whose verdict matched gold. That is the
obvious reading of "don't train on wrong answers", and it starved the minority classes:
of 885 examples, **5** were `Conflicting evidence` and 24 were `Not enough evidence`.
Those are exactly the claims CRAG gets wrong, so they were the ones filtered out. A model
trained on that distribution cannot learn the classes, and the fine-tuned rows would have
shown fine-tuning making the system worse on them.

`--gold-targets` keeps every labelled trajectory and teaches the true label. The
(claim, evidence) pair is still real work the agent did; only the answer is corrected.
The deviation from ExpRAG, which trains on trajectories as produced, is deliberate.

Four adapters exist after this notebook: the two filtered ones from before, and the two
below. The comparison between them is itself a result.


In [ ]:
import os
import pathlib
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/TERABYTE05/VeriMem.git", str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))

RESULTS = "/kaggle/working/results"
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_RESULTS_DIR"] = RESULTS
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## Inputs, checked before any GPU time

Gold targets only help if the **failures** are here too -- they are where the minority
classes live. `experience.generate` writes them to a separate file, so a run that quietly
found only `trajectories.jsonl` would retrain the same 885 examples and look like it had
worked. This cell fails in two minutes rather than discovering it two hours in.

In [ ]:
import pathlib
import shutil
from collections import Counter
import json

TRAJ = pathlib.Path("/kaggle/working/data/trajectories")
TRAJ.mkdir(parents=True, exist_ok=True)

WANT = ("trajectories.jsonl", "trajectories_failed.jsonl")
found = {}
for name in WANT:
    for hit in pathlib.Path("/kaggle/input").rglob(name):
        shutil.copy(hit, TRAJ / name)
        found[name] = sum(1 for _ in open(TRAJ / name))
        print(f"{name:28s} {found[name]:5d}  <- {hit.parent}")
        break

missing = [n for n in WANT if n not in found]
if missing:
    print("\navailable under /kaggle/input:")
    for p in sorted(pathlib.Path("/kaggle/input").rglob("*.jsonl"))[:40]:
        print("   ", p)
    raise SystemExit(
        f"missing {missing}. Gold targets need the failures -- that is where the "
        "minority classes are. Attach the trajectories kernel that wrote both files."
    )

# What the label distribution actually looks like, versus what filtering reported.
paths = [TRAJ / n for n in WANT]
gold = Counter()
agent = Counter()
for p in paths:
    for line in open(p):
        d = json.loads(line)
        if d.get("gold_label"):
            gold[d["gold_label"]] += 1
            if d["gold_label"] == d.get("verdict"):
                agent[d["verdict"]] += 1

total = sum(gold.values())
print(f"\n{total} labelled trajectories\n")
print(f"{'class':<22}{'gold (new)':>12}{'kept (old)':>12}")
for k in sorted(gold, key=lambda k: -gold[k]):
    print(f"{k:<22}{gold[k]:>12}{agent.get(k, 0):>12}")

## Dataset check

`--dry-run` needs no GPU. The verdict counts here are the whole point of the change, so
read them before spending the session on training.

In [ ]:
DATA = ["/kaggle/working/data/trajectories/trajectories.jsonl",
        "/kaggle/working/data/trajectories/trajectories_failed.jsonl"]

for mode in ("plain", "retrieval"):
    print(f"--- {mode} (gold targets) ---")
    subprocess.run(
        [sys.executable, "-m", "training.sft", "--data", *DATA,
         "--name", f"check-gold-{mode}", "--mode", mode,
         "--gold-targets", "--dry-run"],
        env=os.environ, check=True)

## Training dependencies

Unsloth is what makes 4,096-token examples fit: Qwen2.5's ~152k vocab puts the logits
tensor near 2.4 GB at that length, and its fused cross-entropy never materialises it.
Required, not an optimisation.

In [ ]:
!pip install -q peft trl datasets bitsandbytes 2>&1 | tail -2
!pip install -q unsloth unsloth_zoo 2>&1 | tail -2
import torch
print("bf16:", torch.cuda.is_bf16_supported(), "(T4 is False -> fp16, handled automatically)")

## P2.9 (gold) — plain LoRA SFT

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "training.sft", "--data", *DATA,
     "--name", "lora-sft-gold", "--mode", "plain", "--gold-targets"],
    env=os.environ)
if r.returncode != 0:
    raise SystemExit("plain SFT (gold) failed")

## P2.10 (gold) — retrieval-augmented SFT

The same command with `--mode retrieval`. That one flag is the entire ExpRAG comparison,
so nothing else may differ between this cell and the one above.

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "training.sft", "--data", *DATA,
     "--name", "rag-sft-gold", "--mode", "retrieval", "--gold-targets"],
    env=os.environ)
if r.returncode != 0:
    raise SystemExit("retrieval-augmented SFT (gold) failed")

## Results, printed to the log

The log is the reliable output channel: `/kernels/output` pages its file list and the
first page is swamped by thousands of hashed cache entries, so a `metrics.json` deep in
`results/` is effectively unreachable from the API. Everything that matters gets printed.

In [ ]:
import json
import pathlib
import shutil

# Checkpoints are a second copy of each adapter and make the output undownloadable.
for p in pathlib.Path(RESULTS).rglob("checkpoints"):
    shutil.rmtree(p, ignore_errors=True)
shutil.rmtree("/kaggle/working/VeriMem/.git", ignore_errors=True)

for f in sorted(pathlib.Path(RESULTS).rglob("metrics.json")):
    d = json.loads(f.read_text())
    print(f"=== {f.parent.name} ===")
    print(f"  mode {d.get('mode')}  loss {d.get('train_loss')}  runtime {d.get('train_runtime')}")
    print(f"  dataset {json.dumps(d.get('dataset'))}")
    print(f"  adapter {[str(p.relative_to(RESULTS)) for p in f.parent.rglob('adapter_config.json')]}")

print()
!du -sh /kaggle/working/results
!ls -R /kaggle/working/results | head -40